#### Comparison Between Row-based and Parquet Storage

In [1]:
import pandas as pd
import os
import time

In [2]:
# Read CSV
df_csv = pd.read_csv("../data/wearable_data.csv")

In [ ]:
df_csv.head()

In [ ]:
# Convert to Parquet
t0 = time.time()
df_csv.to_parquet(OUTPUT_PARQUET, index=False, compression="snappy")
pq_write_time = time.time() - t0
pq_size_mb = os.path.getsize(OUTPUT_PARQUET) / 1024 / 1024
print(f"  Parquet: {pq_size_mb:.2f} MB  |  write time: {pq_write_time:.2f}s")

print("Conversion completed")

In [3]:
# read parquet file
df_parquet = pd.read_parquet("../data/wearable_data.parquet")
print(df_parquet.head())

            timestamp customer_id     patient_name  heart_rate_bpm  spo2_pct  \
0 2024-03-09 16:00:00      C00494         Mark Cox              63      98.1   
1 2024-03-22 19:00:00      C00040  Brittany Farmer              74      97.9   
2 2024-02-29 12:00:00      C00284   Shelia Wallace              81      97.7   
3 2024-02-14 11:00:00      C00251     Isaiah Avila              80      97.8   
4 2024-03-15 02:00:00      C00028   Stephanie Ross              65      96.6   

   steps  skin_temp_c  hrv_ms  respiratory_rate  activity  
0     22        33.69    71.0                13   working  
1     13        33.92    63.1                15   working  
2     12        34.06    58.0                17   working  
3     12        34.04    58.5                16   working  
4      0        33.82    80.7                15  sleeping  


##### Comapre the file size of parquet file and csv file

In [5]:
# File paths
csv_file = "../data/wearable_data.csv"
parquet_file = "../data/wearable_data.parquet"

# Get file sizes in MB
csv_size = os.path.getsize(csv_file) / (1024 * 1024)
pq_size = os.path.getsize(parquet_file) / (1024 * 1024)

# Print results
print(f"CSV File Size: {csv_size:.2f} MB")
print(f"Parquet File Size: {pq_size:.2f} MB")
print(f"  Size reduction : {(1 - pq_size / csv_size) * 100:.1f}% smaller")
print(f"  Compression ratio: {csv_size / pq_size:.1f}x")

CSV File Size: 71.79 MB
Parquet File Size: 9.57 MB
  Size reduction : 86.7% smaller
  Compression ratio: 7.5x


##### Compare Query Time 
Find patients whose average heart rate is above 90 while sleeping.

In [9]:
start = time.time()

df_csv = pd.read_csv(
    "../data/wearable_data.csv",
    usecols=["customer_id", "activity", "heart_rate_bpm"]
)

result_csv = (
    df_csv[df_csv["activity"] == "sleeping"]
    .groupby("customer_id")["heart_rate_bpm"]
    .mean()
)

result_csv = result_csv[result_csv > 90]

end = time.time()

print(result_csv)

print(f"CSV Query Time: {end - start:.4f} seconds")

Series([], Name: heart_rate_bpm, dtype: float64)
CSV Query Time: 0.4918 seconds


In [8]:
start = time.time()

df_parquet = pd.read_parquet(
    "../data/wearable_data.parquet",
    columns=["customer_id", "activity", "heart_rate_bpm"]
)

result_parquet = (
    df_parquet[df_parquet["activity"] == "sleeping"]
    .groupby("customer_id")["heart_rate_bpm"]
    .mean()
)

result_parquet = result_parquet[result_parquet > 90]

end = time.time()

print(result_parquet)

print(f"Parquet Query Time: {end - start:.4f} seconds")

Series([], Name: heart_rate_bpm, dtype: float64)
Parquet Query Time: 0.1606 seconds
